In [1]:
!pip install -q "unsloth[vllm]" "datasets" "pandas" "tqdm" "vllm" "trl"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
matplotlib 3.8.2 requires numpy<2,>=1.21, but you have numpy 2.2.6 which is incompatible.


In [5]:
!pip uninstall -y numpy scikit-learn opencv-python opencv-python-headless -y
!pip install "numpy<2" scikit-learn opencv-python-headless>=4.11.0

  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (18.0 MB)
Using cached scikit_learn-1.8.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (8.9 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [scikit-learn] [scikit-learn]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
vllm 0.12.0 requires opencv-python-headless>=4.11.0, which is not installed.


In [1]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [2]:
import unsloth
from unsloth import FastLanguageModel
import re
import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

from vllm import LLM, SamplingParams

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
MODEL_NAME = "unsloth/mistral-7b-instruct-v0.3"

SFT_ADAPTER_PATH = "mistral_sft_final"
RL_ADAPTER_PATH  = "mistral_grpo_rl/checkpoint-800"

MAX_SEQ_LENGTH = 2048

In [4]:
# Used for BASE + SFT
SFT_SYSTEM_PROMPT = (
    "You are a math reasoning tutor. "
    "Solve the problem and give the final numeric answer after ####."
)

# Used for RL model ONLY
RL_SYSTEM_PROMPT = """Respond in the following format:
<reasoning>
...
</reasoning>
<answer>
...
</answer>"""

# Answer Extraction

In [5]:
def extract_answer(text):
    if not text:
        return None

    # 1. RL-style <answer>...</answer>
    if "<answer>" in text and "</answer>" in text:
        block = text.split("<answer>")[-1].split("</answer>")[0]
        nums = re.findall(r"-?\d+\.?\d*", block)
        if nums:
            return float(nums[-1])

    # 2. SFT-style ####
    if "####" in text:
        block = text.split("####")[-1]
        nums = re.findall(r"-?\d+\.?\d*", block)
        if nums:
            return float(nums[-1])

    # 3. Fallback: last number
    nums = re.findall(r"-?\d+\.?\d*", text)
    if nums:
        return float(nums[-1])

    return None

In [6]:
def build_prompt(question, tokenizer, mode="sft"):
    system = RL_SYSTEM_PROMPT if mode == "rl" else SFT_SYSTEM_PROMPT
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

# Load Test Split

In [7]:
print("Loading GSM8K test split...")
gsm = load_dataset("openai/gsm8k", "main")
dataset = gsm["test"]

questions = dataset["question"]
answers   = dataset["answer"]

print(f"Total test examples: {len(questions)}")

Loading GSM8K test split...


Total test examples: 1319


In [8]:
_, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
)

# Build prompts
base_prompts = [
    build_prompt(q, tokenizer, mode="sft")
    for q in questions
]

rl_prompts = [
    build_prompt(q, tokenizer, mode="rl")
    for q in questions
]

==((====))==  Unsloth 2025.12.5: Fast Mistral patching. Transformers: 4.57.3. vLLM: 0.12.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.278 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


# Eval Function

In [9]:
import unsloth
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest
import re

def evaluate_with_vllm(
    model_name,
    prompts,
    golds,
    name,
    lora_path=None,
):
    print(f"\n===== Evaluating {name} =====")

    llm = LLM(
        model=model_name,
        dtype="bfloat16",
        tensor_parallel_size=1,
        gpu_memory_utilization=0.75,
        max_model_len=2048,
        enable_lora=(lora_path is not None),
        max_lora_rank=64,
        disable_log_stats=True,
        enforce_eager=True,
    )

    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=512,
    )

    lora_request = None
    if lora_path is not None:
        lora_request = LoRARequest(
            lora_name=name,
            lora_int_id=1,
            lora_path=lora_path,
        )

    outputs = llm.generate(
        prompts,
        sampling_params,
        lora_request=lora_request,
    )

    correct = 0
    for out, gold in zip(outputs, golds):
        if not out.outputs:
            continue

        pred_text = out.outputs[0].text
        pred = extract_answer(pred_text)
        gold = extract_answer(gold)

        if pred is not None and gold is not None and abs(pred - gold) < 1e-4:
            correct += 1

    acc = correct / len(prompts)
    print(f"{name} accuracy: {acc:.2%}")
    return acc

# Run Eval

In [10]:
acc_base = evaluate_with_vllm(
    model_name=MODEL_NAME,
    prompts=base_prompts,
    golds=answers,
    name="base",
)


===== Evaluating base =====
INFO 12-17 09:27:16 [utils.py:253] non-default args: {'dtype': 'bfloat16', 'seed': None, 'max_model_len': 2048, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'unsloth/mistral-7b-instruct-v0.3'}
WARNING 12-17 09:27:16 [arg_utils.py:1175] `seed=None` is equivalent to `seed=0` in V1 Engine. You will no longer be allowed to pass `None` in v0.13.


INFO 12-17 09:27:16 [model.py:637] Resolved architecture: MistralForCausalLM
INFO 12-17 09:27:16 [model.py:1750] Using max model len 2048
INFO 12-17 09:27:22 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 12-17 09:27:22 [vllm.py:601] Enforce eager set, overriding optimization level to -O0
INFO 12-17 09:27:22 [vllm.py:707] Cudagraph is disabled under eager mode


[mistral_common.tokens.tokenizers.utils|WARNING]Multiple valid tokenizer files found. Using tokenizer.model.v3.


WARNING 12-17 09:27:22 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/multiprocessing/spawn.py:132: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  self = reduction.pickle.load(from_parent)


...

(EngineCore_DP0 pid=57111) WARNING 12-17 09:27:51 [vllm.py:608] Inductor compilation was disabled by user settings,Optimizations settings that are only active duringInductor compilation will be ignored.
(EngineCore_DP0 pid=57111) INFO 12-17 09:27:51 [vllm.py:707] Cudagraph is disabled under eager mode
INFO 12-17 09:27:51 [llm.py:343] Supported tasks: ['generate']


Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

base accuracy: 41.17%


[rank0]:[W1217 09:33:01.383631321 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


In [10]:
acc_sft = evaluate_with_vllm(
    model_name=MODEL_NAME,
    prompts=base_prompts,
    golds=answers,
    name="sft",
    lora_path=SFT_ADAPTER_PATH,
)


===== Evaluating sft =====
INFO 12-17 09:36:48 [utils.py:253] non-default args: {'dtype': 'bfloat16', 'seed': None, 'max_model_len': 2048, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'model': 'unsloth/mistral-7b-instruct-v0.3'}
WARNING 12-17 09:36:48 [arg_utils.py:1175] `seed=None` is equivalent to `seed=0` in V1 Engine. You will no longer be allowed to pass `None` in v0.13.


INFO 12-17 09:36:48 [model.py:637] Resolved architecture: MistralForCausalLM
INFO 12-17 09:36:48 [model.py:1750] Using max model len 2048
INFO 12-17 09:36:54 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 12-17 09:36:54 [vllm.py:601] Enforce eager set, overriding optimization level to -O0
INFO 12-17 09:36:54 [vllm.py:707] Cudagraph is disabled under eager mode


[mistral_common.tokens.tokenizers.utils|WARNING]Multiple valid tokenizer files found. Using tokenizer.model.v3.


WARNING 12-17 09:36:54 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/multiprocessing/spawn.py:132: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  self = reduction.pickle.load(from_parent)


...

Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

WARNING 12-17 09:37:24 [input_processor.py:243] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
(EngineCore_DP0 pid=86906) WARNING 12-17 09:37:24 [utils.py:250] Using default LoRA kernel configs


Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

sft accuracy: 44.50%


[rank0]:[W1217 09:40:29.864106589 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


In [10]:
acc_rl = evaluate_with_vllm(
    model_name=MODEL_NAME,
    prompts=rl_prompts,
    golds=answers,
    name="rl",
    lora_path=RL_ADAPTER_PATH,
)


===== Evaluating rl =====
INFO 12-17 09:46:57 [utils.py:253] non-default args: {'dtype': 'bfloat16', 'seed': None, 'max_model_len': 2048, 'gpu_memory_utilization': 0.75, 'disable_log_stats': True, 'enforce_eager': True, 'enable_lora': True, 'max_lora_rank': 64, 'model': 'unsloth/mistral-7b-instruct-v0.3'}
WARNING 12-17 09:46:58 [arg_utils.py:1175] `seed=None` is equivalent to `seed=0` in V1 Engine. You will no longer be allowed to pass `None` in v0.13.
INFO 12-17 09:46:58 [model.py:637] Resolved architecture: MistralForCausalLM
INFO 12-17 09:46:58 [model.py:1750] Using max model len 2048


INFO 12-17 09:47:03 [scheduler.py:228] Chunked prefill is enabled with max_num_batched_tokens=8192.
WARNING 12-17 09:47:03 [vllm.py:601] Enforce eager set, overriding optimization level to -O0
INFO 12-17 09:47:03 [vllm.py:707] Cudagraph is disabled under eager mode


[mistral_common.tokens.tokenizers.utils|WARNING]Multiple valid tokenizer files found. Using tokenizer.model.v3.


WARNING 12-17 09:47:04 [system_utils.py:136] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/multiprocessing/spawn.py:132: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  self = reduction.pickle.load(from_parent)


...

Adding requests:   0%|          | 0/1319 [00:00<?, ?it/s]

WARNING 12-17 09:47:33 [input_processor.py:243] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.
(EngineCore_DP0 pid=119364) WARNING 12-17 09:47:33 [utils.py:250] Using default LoRA kernel configs


Processed prompts:   0%|          | 0/1319 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s…

rl accuracy: 52.46%


[rank0]:[W1217 09:55:14.759295331 ProcessGroupNCCL.cpp:1524] Warning: WARNING: destroy_process_group() was not called before program exit, which can leak resources. For more info, please see https://pytorch.org/docs/stable/distributed.html#shutdown (function operator())


In [12]:
print(f"Base accuracy: {acc_base:.2%}")
print(f"SFT accuracy:  {acc_sft:.2%}")
print(f"RL accuracy:   {acc_rl:.2%}")

Base accuracy: 41.17%
SFT accuracy:  44.50%
RL accuracy:   52.46%
